In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from pydantic import BaseModel, ConfigDict

from vizopt import jaxopt, components
from vizopt.base import ObjectiveTerm, OptimizationProblemTemplate

In [ ]:
random_labels = ["apple", "double espresso", "quiet room", "blue", "green", "yellow", "short", "extremely long"]
n_labels = len(random_labels)
rng = np.random.default_rng(0)
point_positions = 10*rng.random([n_labels, 2])
label_df = pd.DataFrame({"label": random_labels, "x": point_positions[:, 0], "y": point_positions[:, 1]})
label_df["dy"] = 1.
label_df["dx"] = label_df["label"].str.len()*0.5
label_df

In [ ]:
input_parameters = {
    "point_positions": point_positions,
    "rectangle_sizes": label_df[["dx", "dy"]].values,
}
input_parameters

In [ ]:
def initialize(input_parameters):
    return {"rectangle_positions": input_parameters["point_positions"].copy()}

In [ ]:
def plot_rectangles(optim_vars, input_parameters):
    _, ax = plt.subplots(figsize=(4, 3))
    n_rect = optim_vars["rectangle_positions"].shape[0]
    for i_rect in range(n_rect):
        ax.add_patch(plt.Rectangle(
            optim_vars["rectangle_positions"][i_rect, :],
            input_parameters["rectangle_sizes"][i_rect, 0],
            input_parameters["rectangle_sizes"][i_rect, 1],
            alpha=0.5
        ))
        link_x = [optim_vars["rectangle_positions"][i_rect, 0], input_parameters["point_positions"][i_rect, 0]]
        link_y = [optim_vars["rectangle_positions"][i_rect, 1], input_parameters["point_positions"][i_rect, 1]]
        ax.plot(link_x, link_y)
        
    ax.scatter(input_parameters["point_positions"][:, 0], input_parameters["point_positions"][:, 1])

optim_vars = initialize(input_parameters)
plot_rectangles(optim_vars, input_parameters)

In [ ]:
from jax import numpy as jnp


# pairwise intersections between label bounding boxes
def calculate_intersection_loss(optim_vars, input_parameters):
    bbox_matrix = jnp.stack([
        optim_vars["rectangle_positions"],
        optim_vars["rectangle_positions"] + input_parameters["rectangle_sizes"]
    ], axis=1)

    interlabel_inters_matrix = components.multiple_bbox_intersections(bbox_matrix, bbox_matrix)
    interlabel_inters_array = interlabel_inters_matrix[np.triu_indices(interlabel_inters_matrix.shape[0], 1)]
    return jnp.sum(interlabel_inters_array)

#calculate_intersection_loss(optim_vars, input_parameters)

In [ ]:
from pydantic import model_validator


class LabelPositionParams(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    point_positions: np.ndarray  # shape (n, 2)
    rectangle_sizes: np.ndarray  # shape (n, 2)

    @model_validator(mode="after")
    def check_shapes(self) -> "LabelPositionParams":
        for name, arr in [("point_positions", self.point_positions), ("rectangle_sizes", self.rectangle_sizes)]:
            if arr.ndim != 2 or arr.shape[1] != 2:
                raise ValueError(f"{name} must have shape (n, 2), got {arr.shape}")
        if self.point_positions.shape[0] != self.rectangle_sizes.shape[0]:
            raise ValueError(
                f"point_positions and rectangle_sizes must have the same n, "
                f"got {self.point_positions.shape[0]} and {self.rectangle_sizes.shape[0]}"
            )
        return self

In [ ]:
# distances between points and the respective labels
def calculate_point_label_distance_loss(optim_vars, input_parameters):
    return jnp.sum(
        (optim_vars["rectangle_positions"] - input_parameters["point_positions"]) ** 2
    )


label_position_template = OptimizationProblemTemplate(
    terms=[
        ObjectiveTerm(name="intersection_loss", compute=calculate_intersection_loss, multiplier=5.),
        ObjectiveTerm(name="point_label_distance", compute=calculate_point_label_distance_loss),
    ],
    initialize=initialize,
    input_params_class=LabelPositionParams,
    plot_configuration=plot_rectangles,
)

In [ ]:
problem = label_position_template.instantiate(input_parameters)
optim_vars_opt, history = problem.optimize(n_iters=2000, track_every=100)
optim_vars_opt

In [ ]:
problem.plot_configuration(optim_vars_opt, input_parameters)

In [ ]:
_, ax = plt.subplots(figsize=(4, 3))
pd.DataFrame(history).set_index("iteration").plot(ax=ax, marker=".")
ax.set_ylabel("Loss value")
#ax.set_yscale("log")

Next:

* optionally also track optimization variables at different steps and create animation
* initialize with random noise and retry
* dealing with value ranges that need normalization
* refining rectangle label optimization:
    also optimizing attach point...
* derive rectangle size from actual plotted text
* refactor bubblejax in a similar way